<a href="https://colab.research.google.com/github/zsgwu/G5_GWU_CAPST/blob/main/Data_Cleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [6]:
import os

# Path to your project folder
project_path = "/content/drive/MyDrive/group-5/RAG_data"

# Change working directory
os.chdir(project_path)

# Verify
print("Current working directory:")
print(os.getcwd())

print("\nFiles in directory:")
print(os.listdir())

Current working directory:
/content/drive/.shortcut-targets-by-id/1HF_VpvP7ySsREVFXjmHKx9Kq2grdYfhc/group-5/RAG_data

Files in directory:
['OES_Report.xlsx', 'Most-Recent-Cohorts-Field-of-Study.csv', 'cleaned', 'embeddings', 'notebooks', 'CIP2020_SOC2018_Crosswalk.xlsx']


In [7]:
os.makedirs("cleaned", exist_ok=True)
os.makedirs("embeddings", exist_ok=True)
os.makedirs("notebooks", exist_ok=True)

print(os.listdir())

['OES_Report.xlsx', 'Most-Recent-Cohorts-Field-of-Study.csv', 'cleaned', 'embeddings', 'notebooks', 'CIP2020_SOC2018_Crosswalk.xlsx']


In [8]:
import pandas as pd
import numpy as np

In [9]:
oes = pd.read_excel(
    "OES_Report.xlsx",
    sheet_name=0,
    header=5,          # <-- THIS IS THE KEY FIX
    engine="openpyxl"
)

print(oes.columns)

Index(['Occupation (SOC code)', 'Employment (1)',
       'Annual 25th percentile wage (2)', 'Annual median wage (2)'],
      dtype='object')


/usr/local/lib/python3.12/dist-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


In [10]:
# --- Rename columns ---
oes = oes.rename(columns={
    "Occupation (SOC code)": "occupation_raw",
    "Employment (1)": "employment_national",
    "Annual median wage (2)": "annual_median_wage"
})

# --- Drop empty occupation rows ---
oes = oes.dropna(subset=["occupation_raw"])

# --- Extract SOC code ---
oes["soc_code"] = oes["occupation_raw"].str.extract(r"(\d{2}-\d{4})")

# --- Clean occupation title ---
oes["occupation_title"] = (
    oes["occupation_raw"]
    .str.replace(r"\(\d{2}-\d{4}\)", "", regex=True)
    .str.replace("\n", " ")
    .str.strip()
)

# --- Drop rows without SOC ---
oes = oes.dropna(subset=["soc_code"])

# --- Drop aggregate SOC groups ---
oes = oes[
    (oes["soc_code"] != "00-0000") &
    (~oes["soc_code"].str.endswith("0000"))
]

# --- Clean numeric fields ---
for col in ["employment_national", "annual_median_wage"]:
    oes[col] = pd.to_numeric(oes[col], errors="coerce")

# --- Drop rows missing BOTH wage and employment ---
oes = oes.dropna(
    subset=["employment_national", "annual_median_wage"],
    how="all"
)

# --- Add source metadata ---
oes["source"] = "BLS OEWS 2024"

# --- Keep only EduYou columns ---
oes_clean = oes[
    [
        "soc_code",
        "occupation_title",
        "employment_national",
        "annual_median_wage",
        "source"
    ]
].reset_index(drop=True)

# --- Save for RAG ingestion ---
oes_clean.to_csv("cleaned/oes_clean_for_rag.csv", index=False)

print(oes_clean.head(10))
print(f"Final rows: {len(oes_clean)}")

  soc_code                                   occupation_title  \
0  11-1000                                     Top Executives   
1  11-1011                                   Chief Executives   
2  11-1021                    General and Operations Managers   
3  11-1031                                        Legislators   
4  11-2000  Advertising, Marketing, Promotions, Public Rel...   
5  11-2011                Advertising and Promotions Managers   
6  11-2020                       Marketing and Sales Managers   
7  11-2021                                 Marketing Managers   
8  11-2022                                     Sales Managers   
9  11-2030          Public Relations and Fundraising Managers   

   employment_national  annual_median_wage         source  
0            3822780.0            104990.0  BLS OEWS 2024  
1             211850.0            206420.0  BLS OEWS 2024  
2            3584420.0            102950.0  BLS OEWS 2024  
3              26510.0             44810.0  

In [11]:
oes_clean.sample(5)

,soc_code,occupation_title,employment_national,annual_median_wage,source
593,39-3021,Motion Picture Projectionists,1950.0,38180.0,BLS OEWS 2024
155,17-3026,Industrial Engineering Technologists and Techn...,73410.0,64790.0,BLS OEWS 2024
743,47-2022,Stonemasons,8750.0,51990.0,BLS OEWS 2024
766,47-2140,Painters and Paperhangers,225700.0,48660.0,BLS OEWS 2024
109,15-2011,Actuaries,28340.0,125770.0,BLS OEWS 2024


In [12]:
# --- Load College Scorecard data ---
scorecard = pd.read_csv(
    "Most-Recent-Cohorts-Field-of-Study.csv",
    low_memory=False
)

# --- Keep ONLY EduYou-relevant columns ---
scorecard = scorecard[
    [
        "CIPCODE",
        "CIPDESC",
        "CREDDESC",
        "INSTNM",
        "CONTROL",
        "EARN_MDN_4YR_NAT"
    ]
]

# --- Rename columns to stable schema ---
scorecard = scorecard.rename(columns={
    "CIPCODE": "cip_code",
    "CIPDESC": "cip_title",
    "CREDDESC": "degree_level",
    "INSTNM": "institution_name",
    "EARN_MDN_4YR_NAT": "median_earnings_4yr_nat"
})

# --- Clean earnings (drop suppressed / missing values) ---
scorecard["median_earnings_4yr_nat"] = pd.to_numeric(
    scorecard["median_earnings_4yr_nat"],
    errors="coerce"
)

scorecard = scorecard.dropna(subset=["median_earnings_4yr_nat"])

# --- Normalize CIP code format (e.g., 110701 → 11.0701) ---
scorecard["cip_code"] = (
    scorecard["cip_code"]
    .astype(str)
    .str.zfill(6)
    .str[:2] + "." +
    scorecard["cip_code"].astype(str).str.zfill(6).str[2:]
)

# --- Filter to degree levels used in EduYou ---
allowed_degrees = [
    "Bachelor's Degree",
    "Master's Degree"
]

scorecard = scorecard[
    scorecard["degree_level"].isin(allowed_degrees)
]

# --- Add source metadata ---
scorecard["source"] = "College Scorecard"

# --- Final EduYou schema ---
scorecard_clean = scorecard[
    [
        "cip_code",
        "cip_title",
        "degree_level",
        "institution_name",
        "CONTROL",
        "median_earnings_4yr_nat",
        "source"
    ]
].reset_index(drop=True)

# --- Save cleaned file for RAG ingestion ---
scorecard_clean.to_csv(
    "cleaned/college_scorecard_clean_for_rag.csv",
    index=False
)

print(scorecard_clean.head())
print(f"Final rows: {len(scorecard_clean)}")

  cip_code                                          cip_title  \
0  00.0109                                   Animal Sciences.   
1  00.0110                       Food Science and Technology.   
2  00.0110                       Food Science and Technology.   
3  00.0199  Agricultural/Animal/Plant/Veterinary Science a...   
4  00.0199  Agricultural/Animal/Plant/Veterinary Science a...   

        degree_level          institution_name CONTROL  \
0  Bachelor's Degree  Alabama A & M University  Public   
1  Bachelor's Degree  Alabama A & M University  Public   
2    Master's Degree  Alabama A & M University  Public   
3  Bachelor's Degree  Alabama A & M University  Public   
4    Master's Degree  Alabama A & M University  Public   

   median_earnings_4yr_nat             source  
0                  49634.0  College Scorecard  
1                  70873.0  College Scorecard  
2                  88332.0  College Scorecard  
3                  65543.0  College Scorecard  
4                  5

In [13]:
# Load Excel file
xls = pd.ExcelFile(
    "CIP2020_SOC2018_Crosswalk.xlsx",
    engine="openpyxl"
)

# Exclude the File Guide
sheets_to_load = [s for s in xls.sheet_names if s != "File Guide"]

# Load all remaining sheets
crosswalk_sheets = {
    sheet: pd.read_excel(
        "CIP2020_SOC2018_Crosswalk.xlsx",
        sheet_name=sheet,
        engine="openpyxl"
    )
    for sheet in sheets_to_load
}

# Inspect loaded sheets
for name, df in crosswalk_sheets.items():
    print(f"\n=== {name} ===")
    print(df.head())
    print(df.columns)


=== CIP-SOC ===
   CIP2020Code           CIP2020Title SOC2018Code  \
0          1.0  Agriculture, General.     19-1011   
1          1.0  Agriculture, General.     19-1012   
2          1.0  Agriculture, General.     19-1013   
3          1.0  Agriculture, General.     19-4012   
4          1.0  Agriculture, General.     25-1041   

                                    SOC2018Title  
0                              Animal Scientists  
1              Food Scientists and Technologists  
2                      Soil and Plant Scientists  
3                       Agricultural Technicians  
4  Agricultural Sciences Teachers, Postsecondary  
Index(['CIP2020Code', 'CIP2020Title', 'SOC2018Code', 'SOC2018Title'], dtype='object')

=== SOC-CIP ===
  SOC2018Code      SOC2018Title  CIP2020Code  \
0     11-1011  Chief Executives      44.0401   
1     11-1011  Chief Executives      52.0101   
2     11-1011  Chief Executives      52.0201   
3     11-1011  Chief Executives      52.0206   
4     11-1011  

In [14]:
# Use the main CIP → SOC mapping
crosswalk_main = crosswalk_sheets["CIP-SOC"].copy()

print(crosswalk_main.head())
print(crosswalk_main.columns)

   CIP2020Code           CIP2020Title SOC2018Code  \
0          1.0  Agriculture, General.     19-1011   
1          1.0  Agriculture, General.     19-1012   
2          1.0  Agriculture, General.     19-1013   
3          1.0  Agriculture, General.     19-4012   
4          1.0  Agriculture, General.     25-1041   

                                    SOC2018Title  
0                              Animal Scientists  
1              Food Scientists and Technologists  
2                      Soil and Plant Scientists  
3                       Agricultural Technicians  
4  Agricultural Sciences Teachers, Postsecondary  
Index(['CIP2020Code', 'CIP2020Title', 'SOC2018Code', 'SOC2018Title'], dtype='object')


In [16]:
print(crosswalk_main.columns.tolist())

['CIP2020Code', 'CIP2020Title', 'SOC2018Code', 'SOC2018Title']


In [18]:
# --- Start from the CIP-SOC sheet dataframe you already loaded ---
# crosswalk_main = crosswalk_sheets["CIP-SOC"].copy()   # (you already did this)

# Optional: quick check
print(crosswalk_main.columns.tolist())

# --- Rename columns to a stable schema (EduYou) ---
crosswalk_main = crosswalk_main.rename(columns={
    "CIP2020Code": "cip_code",
    "CIP2020Title": "cip_title",
    "SOC2018Code": "soc_code",
    "SOC2018Title": "soc_title"
})

# --- Normalize types + trim whitespace ---
for col in ["cip_code", "cip_title", "soc_code", "soc_title"]:
    crosswalk_main[col] = crosswalk_main[col].astype(str).str.strip()

# --- Replace common “string nulls” with real NaN, then drop missing codes ---
crosswalk_main = crosswalk_main.replace({"": np.nan, "nan": np.nan, "None": np.nan})
crosswalk_main = crosswalk_main.dropna(subset=["cip_code", "soc_code"])

# --- Normalize CIP to 2-digit + '.' + 4-digit (ONLY if not already formatted) ---
# If cip_code already contains '.', we leave it alone.
needs_cip_format = ~crosswalk_main["cip_code"].str.contains(r"\.", na=False)
if needs_cip_format.any():
    # Keep only digits, pad to 6, then insert decimal
    digits = crosswalk_main.loc[needs_cip_format, "cip_code"].str.replace(r"\D", "", regex=True).str.zfill(6)
    crosswalk_main.loc[needs_cip_format, "cip_code"] = digits.str[:2] + "." + digits.str[2:]

# --- Normalize SOC codes to the standard NN-NNNN pattern when possible ---
# (Some rows may contain text; we extract the SOC pattern)
crosswalk_main["soc_code"] = crosswalk_main["soc_code"].str.extract(r"(\d{2}-\d{4})", expand=False)

# Drop rows where SOC extraction failed
crosswalk_main = crosswalk_main.dropna(subset=["soc_code"])

# --- Validate patterns (keep only clean codes) ---
crosswalk_main = crosswalk_main[
    crosswalk_main["cip_code"].str.match(r"^\d{2}\.\d{4}$", na=False) &
    crosswalk_main["soc_code"].str.match(r"^\d{2}-\d{4}$", na=False)
]

# --- Drop aggregate SOC codes (remove 00-0000 and any xx-0000 major groups) ---
crosswalk_main = crosswalk_main[
    (crosswalk_main["soc_code"] != "00-0000") &
    (~crosswalk_main["soc_code"].str.endswith("0000"))
]

# --- De-duplicate exact CIP↔SOC pairs ---
crosswalk_main = crosswalk_main.drop_duplicates(subset=["cip_code", "soc_code"]).reset_index(drop=True)

# --- Keep only EduYou columns (clean output) ---
crosswalk_main = crosswalk_main[["cip_code", "cip_title", "soc_code", "soc_title"]].copy()

# --- Save once (final output for RAG pipeline) ---
crosswalk_main.to_csv("cleaned/cip_soc_crosswalk_clean.csv", index=False)

# --- Sanity checks ---
print(crosswalk_main.head(10))
print(f"Final crosswalk rows: {len(crosswalk_main)}")
print(f"Unique CIP codes: {crosswalk_main['cip_code'].nunique()}")
print(f"Unique SOC codes: {crosswalk_main['soc_code'].nunique()}")

['CIP2020Code', 'CIP2020Title', 'SOC2018Code', 'SOC2018Title']
  cip_code                                          cip_title soc_code  \
0  10.0105              Communications Technology/Technician.  27-3099   
1  10.0105              Communications Technology/Technician.  27-4012   
2  10.0105              Communications Technology/Technician.  27-4014   
3  10.0105              Communications Technology/Technician.  27-4032   
4  10.0201  Photographic and Film/Video Technology/Technic...  27-4011   
5  10.0201  Photographic and Film/Video Technology/Technic...  27-4015   
6  10.0202  Radio and Television Broadcasting Technology/T...  27-4012   
7  10.0202  Radio and Television Broadcasting Technology/T...  27-4015   
8  10.0202  Radio and Television Broadcasting Technology/T...  27-4031   
9  10.0202  Radio and Television Broadcasting Technology/T...  27-4032   

                                       soc_title  
0     Media and Communication Workers, All Other  
1                   

In [19]:
import pandas as pd
import numpy as np

# --------------------------
# Load cleaned datasets
# --------------------------
score = pd.read_csv("cleaned/college_scorecard_clean_for_rag.csv")
oes   = pd.read_csv("cleaned/oes_clean_for_rag.csv")
crosswalk_main = pd.read_csv("cleaned/cip_soc_crosswalk_clean.csv")

# --------------------------
# Normalize join keys (defensive)
# --------------------------
score["cip_code"] = score["cip_code"].astype(str).str.strip()
crosswalk_main["cip_code"] = crosswalk_main["cip_code"].astype(str).str.strip()

oes["soc_code"] = oes["soc_code"].astype(str).str.strip()
crosswalk_main["soc_code"] = crosswalk_main["soc_code"].astype(str).str.strip()

# --------------------------
# 1) Scorecard + Crosswalk (CIP join)
# --------------------------
score_xwalk = score.merge(
    crosswalk_main,
    on="cip_code",
    how="left",
    validate="m:m"   # many-to-many is expected
)

print("After Scorecard↔Crosswalk join:", score_xwalk.shape)
print("Rows missing soc_code (no mapping):", score_xwalk["soc_code"].isna().sum())

# --------------------------
# 2) Add OEWS outcomes (SOC join)
# --------------------------
full_join = score_xwalk.merge(
    oes[["soc_code", "occupation_title", "employment_national", "annual_median_wage"]],
    on="soc_code",
    how="left"
)

print("After adding OEWS:", full_join.shape)
print("Rows missing annual_median_wage:", full_join["annual_median_wage"].isna().sum())

# --------------------------
# Diagnostics: do SOC codes overlap?
# --------------------------
overlap = set(crosswalk_main["soc_code"]).intersection(set(oes["soc_code"]))
print("SOC overlap count (crosswalk ∩ OEWS):", len(overlap))

# Peek at a few joined rows
display(full_join.head(10))

After Scorecard↔Crosswalk join: (87549, 10)
Rows missing soc_code (no mapping): 87549
After adding OEWS: (87549, 13)
Rows missing annual_median_wage: 87549
SOC overlap count (crosswalk ∩ OEWS): 791


,cip_code,cip_title_x,degree_level,institution_name,CONTROL,median_earnings_4yr_nat,source,cip_title_y,soc_code,soc_title,occupation_title,employment_national,annual_median_wage
0,0.0109,Animal Sciences.,Bachelor's Degree,Alabama A & M University,Public,49634.0,College Scorecard,NaN,NaN,NaN,NaN,NaN,NaN
1,0.011,Food Science and Technology.,Bachelor's Degree,Alabama A & M University,Public,70873.0,College Scorecard,NaN,NaN,NaN,NaN,NaN,NaN
2,0.011,Food Science and Technology.,Master's Degree,Alabama A & M University,Public,88332.0,College Scorecard,NaN,NaN,NaN,NaN,NaN,NaN
3,0.0199,Agricultural/Animal/Plant/Veterinary Science a...,Bachelor's Degree,Alabama A & M University,Public,65543.0,College Scorecard,NaN,NaN,NaN,NaN,NaN,NaN
4,0.0199,Agricultural/Animal/Plant/Veterinary Science a...,Master's Degree,Alabama A & M University,Public,58313.0,College Scorecard,NaN,NaN,NaN,NaN,NaN,NaN
5,0.0305,Forestry.,Bachelor's Degree,Alabama A & M University,Public,58784.0,College Scorecard,NaN,NaN,NaN,NaN,NaN,NaN
6,0.0403,"City/Urban, Community, and Regional Planning.",Bachelor's Degree,Alabama A & M University,Public,66874.0,College Scorecard,NaN,NaN,NaN,NaN,NaN,NaN
7,0.0403,"City/Urban, Community, and Regional Planning.",Master's Degree,Alabama A & M University,Public,79665.0,College Scorecard,NaN,NaN,NaN,NaN,NaN,NaN
8,0.0901,Communication and Media Studies.,Master's Degree,Alabama A & M University,Public,71953.0,College Scorecard,NaN,NaN,NaN,NaN,NaN,NaN
9,0.1002,Audiovisual Communications Technologies/Techni...,Bachelor's Degree,Alabama A & M University,Public,44889.0,College Scorecard,NaN,NaN,NaN,NaN,NaN,NaN


In [20]:
print("OEWS rows:", len(oes))
print("Example OEWS SOC codes:", oes["soc_code"].dropna().unique()[:15])

# How many OEWS SOC codes end in 0000 (aggregate groups)?
print("OEWS aggregate SOC rows:", (oes["soc_code"].str.endswith("0000")).sum())

# How many crosswalk SOC codes end in 0000 (should be ~0 after cleaning)?
print("Crosswalk aggregate SOC rows:", (crosswalk_main["soc_code"].str.endswith("0000")).sum())

OEWS rows: 1082
Example OEWS SOC codes: ['11-1000' '11-1011' '11-1021' '11-1031' '11-2000' '11-2011' '11-2020'
 '11-2021' '11-2022' '11-2030' '11-2032' '11-2033' '11-3000' '11-3010'
 '11-3012']
OEWS aggregate SOC rows: 0
Crosswalk aggregate SOC rows: 0


In [21]:
## Join CLEANED DATASETS

score = pd.read_csv("cleaned/college_scorecard_clean_for_rag.csv")
oes   = pd.read_csv("cleaned/oes_clean_for_rag.csv")
crosswalk_main = pd.read_csv("cleaned/cip_soc_crosswalk_clean.csv")

# basic cleanup
score["cip_code"] = score["cip_code"].astype(str).str.strip()
oes["soc_code"] = oes["soc_code"].astype(str).str.strip()
crosswalk_main["cip_code"] = crosswalk_main["cip_code"].astype(str).str.strip()
crosswalk_main["soc_code"] = crosswalk_main["soc_code"].astype(str).str.strip()

In [22]:
# ---- Scorecard CIP4 ----
# If score cip_code looks like "00.0109" or "11.0701", remove the dot then take first 4 digits
score["cip4"] = (
    score["cip_code"]
    .str.replace(".", "", regex=False)
    .str.replace(r"\D", "", regex=True)
    .str[:4]
)

# ---- Crosswalk CIP4 ----
crosswalk_main["cip4"] = (
    crosswalk_main["cip_code"]
    .str.replace(".", "", regex=False)
    .str.replace(r"\D", "", regex=True)
    .str[:4]
)

print("Scorecard unique CIP4:", score["cip4"].nunique())
print("Crosswalk unique CIP4:", crosswalk_main["cip4"].nunique())
print("CIP4 overlap:", len(set(score["cip4"]).intersection(set(crosswalk_main["cip4"]))))

Scorecard unique CIP4: 107
Crosswalk unique CIP4: 410
CIP4 overlap: 0


In [23]:
score_xwalk = score.merge(
    crosswalk_main[["cip4", "cip_title", "soc_code", "soc_title"]],
    on="cip4",
    how="left",
    validate="m:m"
)

print("After Scorecard↔Crosswalk:", score_xwalk.shape)
print("Rows missing soc_code:", score_xwalk["soc_code"].isna().sum())

After Scorecard↔Crosswalk: (87549, 11)
Rows missing soc_code: 87549


In [24]:
full_join = score_xwalk.merge(
    oes[["soc_code", "occupation_title", "employment_national", "annual_median_wage"]],
    on="soc_code",
    how="left"
)

print("After adding OEWS:", full_join.shape)
print("Rows missing annual_median_wage:", full_join["annual_median_wage"].isna().sum())

# SOC overlap diagnostic (should be >0 and usually large)
overlap = set(crosswalk_main["soc_code"]).intersection(set(oes["soc_code"]))
print("SOC overlap (crosswalk ∩ OEWS):", len(overlap))

display(full_join.head(10))

After adding OEWS: (87549, 14)
Rows missing annual_median_wage: 87549
SOC overlap (crosswalk ∩ OEWS): 791


,cip_code,cip_title_x,degree_level,institution_name,CONTROL,median_earnings_4yr_nat,source,cip4,cip_title_y,soc_code,soc_title,occupation_title,employment_national,annual_median_wage
0,0.0109,Animal Sciences.,Bachelor's Degree,Alabama A & M University,Public,49634.0,College Scorecard,0010,NaN,NaN,NaN,NaN,NaN,NaN
1,0.011,Food Science and Technology.,Bachelor's Degree,Alabama A & M University,Public,70873.0,College Scorecard,0011,NaN,NaN,NaN,NaN,NaN,NaN
2,0.011,Food Science and Technology.,Master's Degree,Alabama A & M University,Public,88332.0,College Scorecard,0011,NaN,NaN,NaN,NaN,NaN,NaN
3,0.0199,Agricultural/Animal/Plant/Veterinary Science a...,Bachelor's Degree,Alabama A & M University,Public,65543.0,College Scorecard,0019,NaN,NaN,NaN,NaN,NaN,NaN
4,0.0199,Agricultural/Animal/Plant/Veterinary Science a...,Master's Degree,Alabama A & M University,Public,58313.0,College Scorecard,0019,NaN,NaN,NaN,NaN,NaN,NaN
5,0.0305,Forestry.,Bachelor's Degree,Alabama A & M University,Public,58784.0,College Scorecard,0030,NaN,NaN,NaN,NaN,NaN,NaN
6,0.0403,"City/Urban, Community, and Regional Planning.",Bachelor's Degree,Alabama A & M University,Public,66874.0,College Scorecard,0040,NaN,NaN,NaN,NaN,NaN,NaN
7,0.0403,"City/Urban, Community, and Regional Planning.",Master's Degree,Alabama A & M University,Public,79665.0,College Scorecard,0040,NaN,NaN,NaN,NaN,NaN,NaN
8,0.0901,Communication and Media Studies.,Master's Degree,Alabama A & M University,Public,71953.0,College Scorecard,0090,NaN,NaN,NaN,NaN,NaN,NaN
9,0.1002,Audiovisual Communications Technologies/Techni...,Bachelor's Degree,Alabama A & M University,Public,44889.0,College Scorecard,0100,NaN,NaN,NaN,NaN,NaN,NaN


In [25]:
import pandas as pd
import numpy as np

# Reload RAW scorecard correctly (CIPCODE must be string!)
score_raw = pd.read_csv(
    "Most-Recent-Cohorts-Field-of-Study.csv",
    dtype={"CIPCODE": "string"},
    low_memory=False
)

# Keep only what we need for joining + outcomes
score = score_raw[["CIPCODE", "CIPDESC", "CREDDESC", "INSTNM", "CONTROL", "EARN_MDN_4YR_NAT"]].copy()

# Clean earnings
score["EARN_MDN_4YR_NAT"] = pd.to_numeric(score["EARN_MDN_4YR_NAT"], errors="coerce")
score = score.dropna(subset=["EARN_MDN_4YR_NAT"])

# Create CIP4 from CIPCODE (4-digit, keep leading zeros)
# CIPCODE in this file is typically 4 digits like '0109', '1101', '5202' [3](https://worldbankgroup-my.sharepoint.com/personal/zsultani_worldbank_org/_layouts/15/Doc.aspx?sourcedoc=%7B2A9F8215-1C4D-4D0C-9997-D9347B888827%7D&file=Most-Recent-Cohorts-Field-of-Study.csv&action=default&mobileredirect=true)
score["cip4"] = (
    score["CIPCODE"]
    .str.replace(r"\D", "", regex=True)
    .str.zfill(4)
    .str[:4]
)

# Rename to your EduYou schema (optional)
score = score.rename(columns={
    "CIPDESC": "cip_title",
    "CREDDESC": "degree_level",
    "INSTNM": "institution_name",
    "EARN_MDN_4YR_NAT": "median_earnings_4yr_nat"
})

print(score[["CIPCODE","cip4","cip_title"]].head(10))
print("Scorecard unique cip4:", score["cip4"].nunique())

   CIPCODE  cip4                                          cip_title
0     0109  0109                                   Animal Sciences.
1     0110  0110                       Food Science and Technology.
2     0110  0110                       Food Science and Technology.
4     0199  0199  Agricultural/Animal/Plant/Veterinary Science a...
5     0199  0199  Agricultural/Animal/Plant/Veterinary Science a...
7     0305  0305                                          Forestry.
8     0403  0403      City/Urban, Community, and Regional Planning.
9     0403  0403      City/Urban, Community, and Regional Planning.
10    0901  0901                   Communication and Media Studies.
11    1002  1002  Audiovisual Communications Technologies/Techni...
Scorecard unique cip4: 365


In [26]:
crosswalk_main = pd.read_csv("cleaned/cip_soc_crosswalk_clean.csv")

crosswalk_main["cip4"] = (
    crosswalk_main["cip_code"]
    .astype(str)
    .str.replace(".", "", regex=False)      # 10.0105 -> 100105
    .str.replace(r"\D", "", regex=True)
    .str.zfill(6)
    .str[:4]                                 # -> 1001
)

print(crosswalk_main[["cip_code","cip4","soc_code"]].head(10))
print("Crosswalk unique cip4:", crosswalk_main["cip4"].nunique())

   cip_code  cip4 soc_code
0   10.0105  1001  27-3099
1   10.0105  1001  27-4012
2   10.0105  1001  27-4014
3   10.0105  1001  27-4032
4   10.0201  1002  27-4011
5   10.0201  1002  27-4015
6   10.0202  1002  27-4012
7   10.0202  1002  27-4015
8   10.0202  1002  27-4031
9   10.0202  1002  27-4032
Crosswalk unique cip4: 410


In [27]:
overlap_cip4 = set(score["cip4"]).intersection(set(crosswalk_main["cip4"]))
print("CIP4 overlap count:", len(overlap_cip4))

# show a few unmatched examples from scorecard
unmatched = sorted(set(score["cip4"]) - set(crosswalk_main["cip4"]))[:20]
print("Example unmatched scorecard cip4:", unmatched)

CIP4 overlap count: 318
Example unmatched scorecard cip4: ['0100', '0101', '0102', '0103', '0104', '0105', '0106', '0107', '0108', '0109', '0110', '0111', '0112', '0113', '0180', '0181', '0183', '0199', '0301', '0302']


In [28]:
score_xwalk = score.merge(
    crosswalk_main[["cip4","cip_code","cip_title","soc_code","soc_title"]],
    on="cip4",
    how="left"
)

print("After Scorecard↔Crosswalk:", score_xwalk.shape)
print("Rows missing soc_code:", score_xwalk["soc_code"].isna().sum())

display(score_xwalk.head(10))

After Scorecard↔Crosswalk: (3926204, 11)
Rows missing soc_code: 10858


,CIPCODE,cip_title_x,degree_level,institution_name,CONTROL,median_earnings_4yr_nat,cip4,cip_code,cip_title_y,soc_code,soc_title
0,0109,Animal Sciences.,Bachelor's Degree,Alabama A & M University,Public,49634.0,0109,NaN,NaN,NaN,NaN
1,0110,Food Science and Technology.,Bachelor's Degree,Alabama A & M University,Public,70873.0,0110,NaN,NaN,NaN,NaN
2,0110,Food Science and Technology.,Master's Degree,Alabama A & M University,Public,88332.0,0110,NaN,NaN,NaN,NaN
3,0199,Agricultural/Animal/Plant/Veterinary Science a...,Bachelor's Degree,Alabama A & M University,Public,65543.0,0199,NaN,NaN,NaN,NaN
4,0199,Agricultural/Animal/Plant/Veterinary Science a...,Master's Degree,Alabama A & M University,Public,58313.0,0199,NaN,NaN,NaN,NaN
5,0305,Forestry.,Bachelor's Degree,Alabama A & M University,Public,58784.0,0305,NaN,NaN,NaN,NaN
6,0403,"City/Urban, Community, and Regional Planning.",Bachelor's Degree,Alabama A & M University,Public,66874.0,0403,NaN,NaN,NaN,NaN
7,0403,"City/Urban, Community, and Regional Planning.",Master's Degree,Alabama A & M University,Public,79665.0,0403,NaN,NaN,NaN,NaN
8,0901,Communication and Media Studies.,Master's Degree,Alabama A & M University,Public,71953.0,0901,NaN,NaN,NaN,NaN
9,1002,Audiovisual Communications Technologies/Techni...,Bachelor's Degree,Alabama A & M University,Public,44889.0,1002,10.0201,Photographic and Film/Video Technology/Technic...,27-4011,Audio and Video Technicians


In [29]:
oes = pd.read_csv("cleaned/oes_clean_for_rag.csv")
oes["soc_code"] = oes["soc_code"].astype(str).str.strip()

full_join = score_xwalk.merge(
    oes[["soc_code","occupation_title","employment_national","annual_median_wage"]],
    on="soc_code",
    how="left"
)

print("After adding OEWS:", full_join.shape)
print("Rows missing annual_median_wage:", full_join["annual_median_wage"].isna().sum())

display(full_join.head(10))

After adding OEWS: (3926204, 14)
Rows missing annual_median_wage: 194191


,CIPCODE,cip_title_x,degree_level,institution_name,CONTROL,median_earnings_4yr_nat,cip4,cip_code,cip_title_y,soc_code,soc_title,occupation_title,employment_national,annual_median_wage
0,0109,Animal Sciences.,Bachelor's Degree,Alabama A & M University,Public,49634.0,0109,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,0110,Food Science and Technology.,Bachelor's Degree,Alabama A & M University,Public,70873.0,0110,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,0110,Food Science and Technology.,Master's Degree,Alabama A & M University,Public,88332.0,0110,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,0199,Agricultural/Animal/Plant/Veterinary Science a...,Bachelor's Degree,Alabama A & M University,Public,65543.0,0199,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,0199,Agricultural/Animal/Plant/Veterinary Science a...,Master's Degree,Alabama A & M University,Public,58313.0,0199,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,0305,Forestry.,Bachelor's Degree,Alabama A & M University,Public,58784.0,0305,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,0403,"City/Urban, Community, and Regional Planning.",Bachelor's Degree,Alabama A & M University,Public,66874.0,0403,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,0403,"City/Urban, Community, and Regional Planning.",Master's Degree,Alabama A & M University,Public,79665.0,0403,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,0901,Communication and Media Studies.,Master's Degree,Alabama A & M University,Public,71953.0,0901,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,1002,Audiovisual Communications Technologies/Techni...,Bachelor's Degree,Alabama A & M University,Public,44889.0,1002,10.0201,Photographic and Film/Video Technology/Technic...,27-4011,Audio and Video Technicians,Audio and Video Technicians,70080.0,54830.0


RAG Friendly join

In [41]:
import os
import numpy as np

# Check folder
print("Working dir:", os.getcwd())
print("Top-level files:", os.listdir())

os.makedirs("cleaned", exist_ok=True)
print("\nFiles in cleaned/:", sorted(os.listdir("cleaned")))

# Define mystery SOCs (present but wage missing)
mystery_socs = {
    '27-2011','27-2031','27-2042','27-2091','27-2099',
    '29-1022','29-1023','29-1024','29-1211','29-1212','29-1213','29-1214',
    '29-1217','29-1218','29-1222','29-1223','29-1224','29-1229',
    '29-1241','29-1242','29-1243','29-1249'
}

# Add wage status to OEWS table
oes["wage_status"] = np.where(
    oes["annual_median_wage"].notna(), "ok",
    np.where(oes["soc_code"].isin(mystery_socs), "present_but_missing_wage", "missing_wage")
)
oes["in_oes"] = True

# Add diagnostics to joined table
full_join["soc_in_oes"] = full_join["soc_code"].isin(soc_in_oes)
full_join["wage_available"] = full_join["annual_median_wage"].notna()

print("Rows with no SOC (crosswalk join failed):", full_join["soc_code"].isna().sum())
print("Rows with SOC but not in OEWS:",
      ((full_join["soc_code"].notna()) & (~full_join["soc_in_oes"])).sum())
print("Rows with SOC in OEWS but wage missing:",
      ((full_join["soc_code"].notna()) & (full_join["soc_in_oes"]) & (~full_join["wage_available"])).sum())

# Create RAG-ready subset (wage available only)
full_join_rag = full_join.dropna(subset=["annual_median_wage"]).copy()
print("RAG-ready rows:", len(full_join_rag))
print("Unique SOC with wages:", full_join_rag["soc_code"].nunique())

Working dir: /content/drive/.shortcut-targets-by-id/1HF_VpvP7ySsREVFXjmHKx9Kq2grdYfhc/group-5/RAG_data
Top-level files: ['OES_Report.xlsx', 'Most-Recent-Cohorts-Field-of-Study.csv', 'cleaned', 'embeddings', 'notebooks', 'CIP2020_SOC2018_Crosswalk.xlsx']

Files in cleaned/: ['cip_soc_crosswalk_clean.csv', 'college_scorecard_clean_for_rag.csv', 'oes_clean_for_rag.csv']
Rows with no SOC (crosswalk join failed): 10858
Rows with SOC but not in OEWS: 136082
Rows with SOC in OEWS but wage missing: 47251
RAG-ready rows: 3732013
Unique SOC with wages: 597


Save joined files

In [42]:
# Save full join (big file — but complete)
full_join.to_csv("cleaned/eduyou_joined_full.csv", index=False)

# Save RAG-ready join (only rows with wages)
full_join_rag.to_csv("cleaned/eduyou_joined_rag_only.csv", index=False)

print("Saved:")
print(" - cleaned/eduyou_joined_full.csv")
print(" - cleaned/eduyou_joined_rag_only.csv")

Saved:
 - cleaned/eduyou_joined_full.csv
 - cleaned/eduyou_joined_rag_only.csv


save a slim joined file (best for RAG + avoids huge CSVs)

In [43]:
cols_needed = [
    "cip4", "cip_title_x", "degree_level", "institution_name", "CONTROL",
    "median_earnings_4yr_nat",
    "soc_code", "soc_title", "occupation_title",
    "employment_national", "annual_median_wage"
]

# Make sure these columns exist in your full_join (adjust cip_title column name if needed)
slim_join = full_join[cols_needed].copy()

slim_join.to_csv("cleaned/eduyou_joined_for_rag.csv", index=False)

# Also save the wage-only slim version
slim_join_rag = slim_join.dropna(subset=["annual_median_wage"]).copy()
slim_join_rag.to_csv("cleaned/eduyou_joined_for_rag_wage_only.csv", index=False)

print("Saved:")
print(" - cleaned/eduyou_joined_for_rag.csv")
print(" - cleaned/eduyou_joined_for_rag_wage_only.csv")
print("Rows (slim):", len(slim_join))
print("Rows (wage-only):", len(slim_join_rag))

Saved:
 - cleaned/eduyou_joined_for_rag.csv
 - cleaned/eduyou_joined_for_rag_wage_only.csv
Rows (slim): 3926204
Rows (wage-only): 3732013
